### RANSAC Shape detection from scratch using Open3D
https://www.youtube.com/watch?v=BHzQhaxmvEg&list=LL&index=2&ab_channel=FlorentPoux

https://learngeodata.eu/3d-shape-detection-with-ransac-and-python-sphere-and-plane/

In [17]:
import numpy as np
import open3d as o3d
from scipy.optimize import least_squares

In [36]:
# Generate a synthetic point cloud
# 2000 points for our point cloud
num_points = 2000

# Create Plane
plane_points = np.random.randn(num_points // 2, 3) # half of the points are on the plane, 3 dimensions
plane_points[:, 2] = 0.1 * np.random.randn(num_points // 2) + 1 # add some noise to the z-coords

# Create sphere
sphere_center = np.array([2, 2, 2]) # center of the sphere at (2, 2, 2)
sphere_radius = 1.0 # radius of the sphere
theta = np.random.uniform(0, 2 * np.pi, num_points // 2) # random angles
phi = np.random.uniform(0, np.pi, num_points // 2) # random angles

# Generate points on the sphere
x = sphere_radius * np.cos(theta) * np.sin(phi) + sphere_center[0] # (500,)
y = sphere_radius * np.sin(theta) * np.sin(phi) + sphere_center[1] # (500,)
z = sphere_radius * np.cos(phi) + sphere_center[2] # (500,)
sphere_points = np.column_stack((x, y, z)) # stack the points together, (500, 3)
sphere_points += 0.05 * np.random.randn(num_points // 2, 3) # add some noise to the points

# Combine the plane and sphere points
points = np.vstack((plane_points, sphere_points))

# Create a point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

o3d.visualization.draw_geometries([pcd])

In [37]:
# 3D RANSAC functions
def ransac_plane(points, threshold, iterations):
    best_count = 0
    best_model = None
    best_inliers = None
    n_pts = points.shape[0]

    for _ in range(iterations):
        # -- 1) sample 3 random pts
        idx = np.random.choice(n_pts, 3, replace=False)
        p1, p2, p3 = points[idx]
        # -- 2) fit plane through them
        v1 = p2 - p1
        v2 = p3 - p1
        normal = np.cross(v1, v2)
        norm_n = np.linalg.norm(normal)
        if norm_n == 0:
            continue  # colinear points
        normal = normal / norm_n  # unit normal
        a, b, c = normal
        d = -np.dot(normal, p1)
        # -- 3) compute distances & inliers
        distances = np.abs(points.dot(normal) + d)
        inlier_mask = distances < threshold
        count = np.count_nonzero(inlier_mask)
        # -- 4) keep best
        if count > best_count:
            best_count = count
            best_model = (a, b, c, d)
            best_inliers = points[inlier_mask]

    if best_model is None:
        raise RuntimeError("RANSAC failed to find a plane.")
    return best_model, best_inliers

def ransac_sphere(points, threshold, iterations):
    best_count = 0
    best_model = None
    best_inliers = None
    n_pts = points.shape[0]

    for _ in range(iterations):
        # -- 1) sample 4 random pts
        idx = np.random.choice(n_pts, 4, replace=False)
        sample = points[idx]
        # -- 2) set up linear system for sphere center and c:
        #    x^2+y^2+z^2 = [2x,2y,2z,1]·[x0,y0,z0,c]^T
        A = np.hstack((2*sample, np.ones((4,1))))
        B = np.sum(sample**2, axis=1)
        # -- 3) solve
        try:
            x0, y0, z0, c = np.linalg.lstsq(A, B, rcond=None)[0]
        except np.linalg.LinAlgError:
            continue
        # check radius validity
        rad_sq = x0**2 + y0**2 + z0**2 + c
        if rad_sq <= 0:
            continue
        radius = np.sqrt(rad_sq)
        # -- 4) compute inliers
        center = np.array([x0, y0, z0])
        dists = np.abs(np.linalg.norm(points - center, axis=1) - radius)
        inlier_mask = dists < threshold
        count = np.count_nonzero(inlier_mask)
        # -- 5) keep best
        if count > best_count:
            best_count = count
            best_model = (x0, y0, z0, radius)
            best_inliers = points[inlier_mask]

    if best_model is None:
        raise RuntimeError("RANSAC failed to find a sphere.")
    return best_model, best_inliers

In [38]:
# Run function
plane_params, plane_inliers = ransac_plane(points, threshold=0.1, iterations=1000)
print(f"Plane parameters: {plane_params}")
print(f"Plane inliers: {plane_inliers}")

sphere_params, sphere_inliers = ransac_sphere(points, threshold=0.1, iterations=1000)
print(f"Sphere parameters: {sphere_params}")
print(f"Sphere inliers: {sphere_inliers}")

Plane parameters: (np.float64(-0.002005052575624174), np.float64(0.004747868641971619), np.float64(-0.9999867186655668), np.float64(1.022870423319747))
Plane inliers: [[-0.07383564 -0.45238202  0.97131468]
 [-0.23583957  0.78936804  1.06248137]
 [-0.29207427  0.27769881  1.02835175]
 ...
 [ 2.08623657  1.89566199  1.01216161]
 [ 2.03268821  2.01476799  1.02335161]
 [ 1.69545764  1.96455602  1.05010421]]
Sphere parameters: (np.float64(1.9989828896471298), np.float64(2.0400938515347797), np.float64(2.0208277087006277), np.float64(0.9980303091274157))
Sphere inliers: [[1.83906724 1.74039883 1.04955009]
 [1.57621817 2.01770396 1.02879348]
 [2.10864326 1.69211089 1.03753116]
 ...
 [1.61452467 1.49657006 1.21335315]
 [2.7311838  1.57349537 1.41372125]
 [1.65868215 2.88983198 1.67636267]]


In [45]:
# visualize our results

# select and segment the planar points
plane_pc = o3d.geometry.PointCloud()
plane_pc.points = o3d.utility.Vector3dVector(plane_inliers)
plane_pc.paint_uniform_color([1, 0, 0]) # plane = red

# select and segment the spherical points
sphere_pc = o3d.geometry.PointCloud()
sphere_pc.points = o3d.utility.Vector3dVector(sphere_inliers)
sphere_pc.paint_uniform_color([0, 1, 0]) # sphere = green

# select and segment the remaining points
other_pc = o3d.geometry.PointCloud()
other_pc.points = o3d.utility.Vector3dVector(points)
other_pc.paint_uniform_color([0, 0, 1]) # other = blue

# Create coordinate frame
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])

# visualize
o3d.visualization.draw_geometries([plane_pc, sphere_pc, other_pc, coordinate_frame])

In [43]:
o3d.visualization.draw_geometries([other_pc])